# Analog Video Restoration on Google Colab — VapourSynth R76

End-to-end pipeline that mirrors the **Windows** master script `masterpaldvcTOP.vpy`:

```
src(AVI rawvideo)  ->  YUV444P8 (Rec.601, error_diffusion)
                    ->  QTGMC deinterlace (TFF, GPU edge step)
                    ->  Crop 720x576 -> 708x570
                    ->  KNLMeansCL denoise (GPU)
                    ->  fine_dehalo (vsdehalo)
                    ->  Rec.601 -> Rec.709 (matrix recompute via RGBS)
                    ->  nnedi3_rpow2 upscale to 1440x1080 (GPU)
                    ->  CAS sharpen
                    ->  vspipe y4m | ffmpeg libx264 / NVENC + AAC mux
```

**Why this notebook works where the 2021 one didn't**

The 2021 notebook installed VapourSynth via `apt`, pulling a `vspipe` linked
against `libpython3.6m`. Each time Colab bumped its Python (3.6 -> 3.11 -> 3.12)
it broke; the desperate `os.symlink` fix produced `undefined symbol:
getVSScriptAPI`. When VS failed, the notebook silently fell back to a plain
`ffmpeg yadif=1:1,hqdn3d,scale,unsharp` chain — that is NOT the pipeline.

The 2026 fix: VapourSynth R76 ships an **abi3** wheel
(`vapoursynth-76-cp312-abi3-manylinux_2_28_x86_64`). abi3 = stable ABI for every
CPython >= 3.12. We bundle our own Python 3.12 + VS R76 + all 20-odd plugins
into one `vs-portable-linux-py312.tar.zst`, built once by `Dockerfile`, then
just `wget` + `tar` it into `/opt/vs` at the start of each Colab session.

If a bundle URL is set in Cell 2, we use it (fast, ~20 s setup). If not, the
notebook falls back to building Python 3.12 + VapourSynth + plugins inside the
Colab session (slow, ~10-20 min, but still produces the SAME pipeline).


In [ ]:
#@title 1 - GPU + env probe + pick GPU mode
#@markdown Runs `nvidia-smi`, writes the NVIDIA OpenCL ICD, runs `clinfo`,
#@markdown and sets `VS_GPU_MODE` env var to `opencl` (preferred) or `cuda`.
import os, subprocess, sys

def run(cmd, **kw):
    return subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)

print("=== nvidia-smi ===")
r = run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
print(r.stdout or r.stderr)

print("=== python / os / glibc ===")
print(sys.version)
print(run("cat /etc/os-release | grep -E 'PRETTY_NAME|VERSION_ID'").stdout)
print(run("ldd --version | head -1").stdout)

print("=== installing clinfo ===")
run("apt-get -qq install -y clinfo")

# Register NVIDIA's OpenCL ICD so clinfo / VS plugins see the GPU.
os.makedirs("/etc/OpenCL/vendors", exist_ok=True)
with open("/etc/OpenCL/vendors/nvidia.icd", "w") as f:
    f.write("libnvidia-opencl.so.1\n")

print("=== clinfo -l ===")
clinfo = run("clinfo -l").stdout
print(clinfo or "(empty)")

has_gpu = any(s in clinfo for s in ("NVIDIA", "CUDA", "Tesla", "GeForce", "L4", "A100"))
if has_gpu:
    os.environ["VS_GPU_MODE"] = "opencl"
    print("\n>>> VS_GPU_MODE = opencl  (full pipeline, matches Windows .vpy)")
else:
    os.environ["VS_GPU_MODE"] = "cuda"
    print("\n>>> VS_GPU_MODE = cuda    (CPU znedi3 / hqdn3d fallback path)")

# Persist for shell subprocesses.
with open("/etc/profile.d/vs_gpu_mode.sh", "w") as f:
    f.write(f'export VS_GPU_MODE={os.environ["VS_GPU_MODE"]}\n')


In [ ]:
#@title 2 - Fetch + unpack prebuilt VS bundle (~20s) OR build in-session
#@markdown ### Path A (fast, recommended): set `BUNDLE_URL` to your GitHub Release asset URL.
#@markdown ### Path B (no URL set): rebuilds Python 3.12 + VS R76 + plugins inside Colab (~10-20 min).
BUNDLE_URL = ""  #@param {type:"string"}

import os, subprocess, sys, base64, textwrap
from pathlib import Path

VSPREFIX = "/opt/vs"
os.environ["VSPREFIX"] = VSPREFIX

def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise SystemExit(f"FAILED ({r.returncode}): {cmd}")

if BUNDLE_URL:
    # ---------- Path A: prebuilt bundle ------------------------------------
    print(f"=== Path A: downloading {BUNDLE_URL}")
    sh("apt-get -qq install -y zstd wget")
    sh(f"wget -q -O /tmp/vs.tar.zst '{BUNDLE_URL}'")
    sh(f"mkdir -p {VSPREFIX} && tar --zstd -xf /tmp/vs.tar.zst -C /opt")
    sh(f"cat {VSPREFIX}/BUNDLE_VERSION 2>/dev/null || echo '(no manifest)'")
else:
    # ---------- Path B: build in Colab session -----------------------------
    print("=== Path B: BUNDLE_URL not set; building VS R76 + plugins in-session")
    print("    (10-20 minutes; results survive only the lifetime of this Colab VM)")

    sh("apt-get -qq update")
    sh("apt-get -qq install -y software-properties-common")
    sh("add-apt-repository -y ppa:deadsnakes/ppa")
    sh("apt-get -qq update")
    sh("apt-get -qq install -y build-essential gcc-11 g++-11 "
       "cmake meson ninja-build pkg-config nasm yasm autoconf automake libtool "
       "patchelf file xz-utils unzip p7zip-full git wget zstd "
       "libfftw3-dev libzimg-dev zlib1g-dev "
       "libavcodec-dev libavformat-dev libavutil-dev libswscale-dev libswresample-dev "
       "ocl-icd-opencl-dev opencl-headers")
    sh("update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-11 100")
    sh("update-alternatives --install /usr/bin/g++ g++ /usr/bin/g++-11 100")

    # Bundled RELOCATABLE Python via python-build-standalone (NOT venv).
    # A venv only symlinks to the system interpreter; pbs ships a real
    # relocatable CPython so the result matches the Docker oven exactly.
    PBS_URL = ("https://github.com/astral-sh/python-build-standalone/releases/"
               "download/20260602/cpython-3.12.13%2B20260602-"
               "x86_64-unknown-linux-gnu-install_only.tar.gz")
    sh(f"cd /tmp && wget -qO pbs.tar.gz '{PBS_URL}' && mkdir -p {VSPREFIX} && tar xzf pbs.tar.gz && mv python {VSPREFIX}/py && rm -f pbs.tar.gz")
    sh(f"{VSPREFIX}/py/bin/python -m pip install --upgrade pip wheel setuptools cython numpy vapoursynth==76")
    sh(f"mkdir -p {VSPREFIX}/plugins {VSPREFIX}/lib {VSPREFIX}/bin")

    # Decode the embedded build_plugins.sh and run it (same script Docker uses).
    BUILD_PLUGINS_B64 = "IyEvdXNyL2Jpbi9lbnYgYmFzaAojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgYnVpbGRfcGx1Z2lucy5zaCAtLSBidWlsZCBWYXBvdXJTeW50aCBwbHVnaW4gLnNvIHNldCBpbnRvICR7VlNQUkVGSVh9L3BsdWdpbnMKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIFR3by10aWVyIHBvbGljeToKIyAgIFJFUVVJUkVEIHBsdWdpbnMgLS0gdGhlIHBpcGVsaW5lIGNhbm5vdCBydW4gd2l0aG91dCB0aGVtLiBGYWlsdXJlIHN0b3BzCiMgICAgICAgICAgICAgICAgICAgICAgIHRoZSBlbnRpcmUgYnVpbGQgd2l0aCBub24temVybyBleGl0LgojICAgT1BUSU9OQUwgcGx1Z2lucyAtLSBuaWNlLXRvLWhhdmUgaGVscGVycyAobW9zdGx5IFFUR01DIGNvcm5lci1jYXNlcyBvcgojICAgICAgICAgICAgICAgICAgICAgICBmYWxsYmFjayBkZW5vaXNlcnMpLiBGYWlsdXJlcyBhcmUgbG9nZ2VkIGJ1dCB0aGUgYnVpbGQKIyAgICAgICAgICAgICAgICAgICAgICAgY29udGludWVzLiBUaGUgYnVuZGxlIGlzIHN0aWxsIHVzZWZ1bC4KIwojIFNtYXJ0IGNsb25lOiB0cnkgYG1hc3RlcmAsIHRoZW4gYG1haW5gLCB0aGVuIGRlZmF1bHQgSEVBRC4gUmVwb3MgbW92ZS4KIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpzZXQgLXVvIHBpcGVmYWlsCgo6ICIke1ZTUFJFRklYOj0vb3B0L3ZzfSIKUExVR0lORElSPSIke1ZTUFJFRklYfS9wbHVnaW5zIgpMT0dESVI9Ii90bXAvcGx1Z2luLWxvZ3MiCm1rZGlyIC1wICIke1BMVUdJTkRJUn0iICIke0xPR0RJUn0iCgpleHBvcnQgUEtHX0NPTkZJR19QQVRIPSIke1ZTUFJFRklYfS9saWIvcGtnY29uZmlnOiR7UEtHX0NPTkZJR19QQVRIOi19IgpleHBvcnQgQ0ZMQUdTPSItTzMgLWZQSUMgJHtDRkxBR1M6LX0iCmV4cG9ydCBDWFhGTEFHUz0iLU8zIC1mUElDICR7Q1hYRkxBR1M6LX0iCmV4cG9ydCBMRF9MSUJSQVJZX1BBVEg9IiR7VlNQUkVGSVh9L2xpYjoke0xEX0xJQlJBUllfUEFUSDotfSIKCldPUks9L3RtcC92c2J1aWxkCm1rZGlyIC1wICIke1dPUkt9IgpjZCAiJHtXT1JLfSIKCkZBSUxFRF9SRVFVSVJFRD0oKQpGQUlMRURfT1BUSU9OQUw9KCkKQlVJTFQ9KCkKCiMgLS0tLSBoZWxwZXJzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCiMgU21hcnQgY2xvbmU6IGdjbG9uZSBSRVBPX1VSTCBESVJOQU1FIC0tIHRyaWVzIG1hc3RlciwgdGhlbiBtYWluLCB0aGVuIGRlZmF1bHQuCmdjbG9uZSgpIHsKICAgIGxvY2FsIHVybD0iJDEiIGRzdD0iJDIiCiAgICBbIC1kICIke2RzdH0iIF0gJiYgcm0gLXJmICIke2RzdH0iCiAgICBnaXQgY2xvbmUgLS1kZXB0aCAxIC0tYnJhbmNoIG1hc3RlciAiJHt1cmx9IiAiJHtkc3R9IiAyPi9kZXYvbnVsbCBcCiAgICAgIHx8IGdpdCBjbG9uZSAtLWRlcHRoIDEgLS1icmFuY2ggbWFpbiAiJHt1cmx9IiAiJHtkc3R9IiAyPi9kZXYvbnVsbCBcCiAgICAgIHx8IGdpdCBjbG9uZSAtLWRlcHRoIDEgIiR7dXJsfSIgIiR7ZHN0fSIKfQoKIyBDb3B5IGFueSBidWlsdCAuc28gZmlsZXMgaW4gYSBkaXJlY3RvcnkgdHJlZSBpbnRvICR7UExVR0lORElSfS4KaW5zdF9zbygpIHsKICAgIGZpbmQgIiQxIiAtbmFtZSAnKi5zbycgLXR5cGUgZiAtZXhlYyBjcCAtdiB7fSAiJHtQTFVHSU5ESVJ9LyIgXDsgMj4vZGV2L251bGwgfHwgdHJ1ZQp9CgptZXNvbl9idWlsZCgpIHsKICAgIGxvY2FsIHNyYz0iJDEiOyBzaGlmdAogICAgcm0gLXJmICIke3NyY30vYnVpbGQiCiAgICBtZXNvbiBzZXR1cCAiJHtzcmN9L2J1aWxkIiAiJHtzcmN9IiAtLWJ1aWxkdHlwZT1yZWxlYXNlIFwKICAgICAgICAtLXByZWZpeD0iJHtWU1BSRUZJWH0iIC0tbGliZGlyPSIke1ZTUFJFRklYfS9saWIiICIkQCIgXAogICAgICAmJiBuaW5qYSAtQyAiJHtzcmN9L2J1aWxkIiAtaiIkKG5wcm9jKSIgXAogICAgICAmJiAobmluamEgLUMgIiR7c3JjfS9idWlsZCIgaW5zdGFsbCAyPi9kZXYvbnVsbCB8fCB0cnVlKSBcCiAgICAgICYmIGluc3Rfc28gIiR7c3JjfS9idWlsZCIKfQoKY21ha2VfYnVpbGQoKSB7CiAgICBsb2NhbCBzcmM9IiQxIjsgc2hpZnQKICAgIHJtIC1yZiAiJHtzcmN9L2J1aWxkIgogICAgY21ha2UgLVMgIiR7c3JjfSIgLUIgIiR7c3JjfS9idWlsZCIgLUdOaW5qYSBcCiAgICAgICAgLURDTUFLRV9CVUlMRF9UWVBFPVJlbGVhc2UgXAogICAgICAgIC1EQ01BS0VfSU5TVEFMTF9QUkVGSVg9IiR7VlNQUkVGSVh9IiBcCiAgICAgICAgLURDTUFLRV9JTlNUQUxMX0xJQkRJUj1saWIgXAogICAgICAgICIkQCIgXAogICAgICAmJiBjbWFrZSAtLWJ1aWxkICIke3NyY30vYnVpbGQiIC1qIiQobnByb2MpIiBcCiAgICAgICYmIChjbWFrZSAtLWluc3RhbGwgIiR7c3JjfS9idWlsZCIgMj4vZGV2L251bGwgfHwgdHJ1ZSkgXAogICAgICAmJiBpbnN0X3NvICIke3NyY30vYnVpbGQiCn0KCiMgYnVpbGQgTkFNRSBSRVFVSVJFRF9GTEFHICBDTUQgLi4uCiMgICBOQU1FOiAgICAgICAgICAgbGFiZWwgdXNlZCBpbiBsb2dzCiMgICBSRVFVSVJFRF9GTEFHOiAgInJlcSIgb3IgIm9wdCIKIyAgIENNRCAuLi46ICAgICAgICB0aGUgYWN0dWFsIGJ1aWxkIGNvbW1hbmRzIChydW4gYXMgYmFzaCBzdWJzaGVsbCkKYnVpbGQoKSB7CiAgICBsb2NhbCBuYW1lPSIkMSIgdGllcj0iJDIiOyBzaGlmdCAyCiAgICBsb2NhbCBiZWZvcmUgYWZ0ZXIgZGVsdGEgbG9nZj0iJHtMT0dESVJ9LyR7bmFtZX0ubG9nIgogICAgYmVmb3JlPSQobHMgLTEgIiR7UExVR0lORElSfSIgMj4vZGV2L251bGwgfCB3YyAtbCkKICAgIGVjaG8KICAgIGVjaG8gIj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PSIKICAgIGVjaG8gIlske3RpZXJeXn1dICR7bmFtZX0iCiAgICBlY2hvICI9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0iCiAgICBpZiAoIHNldCAtZTsgIiRAIiApID4gIiR7bG9nZn0iIDI+JjE7IHRoZW4KICAgICAgICBhZnRlcj0kKGxzIC0xICIke1BMVUdJTkRJUn0iIDI+L2Rldi9udWxsIHwgd2MgLWwpCiAgICAgICAgZGVsdGE9JCgoYWZ0ZXIgLSBiZWZvcmUpKQogICAgICAgIGlmIFsgIiR7ZGVsdGF9IiAtZ3QgMCBdOyB0aGVuCiAgICAgICAgICAgIGVjaG8gIiAgT0sgIChhZGRlZCAke2RlbHRhfSAuc28gZmlsZXMpIgogICAgICAgICAgICBCVUlMVCs9KCIke25hbWV9IikKICAgICAgICBlbHNlCiAgICAgICAgICAgIGVjaG8gIiAgV0FSTjogYnVpbGQgc3VjY2VlZGVkIGJ1dCBubyAuc28gaW5zdGFsbGVkIC0tIGNoZWNrICR7bG9nZn0iCiAgICAgICAgICAgIEJVSUxUKz0oIiR7bmFtZX0gKG5vLXNvKSIpCiAgICAgICAgZmkKICAgIGVsc2UKICAgICAgICBlY2hvICIgIEZBSUwgLS0gbGFzdCAyMCBsaW5lcyBvZiBsb2c6IgogICAgICAgIHRhaWwgLTIwICIke2xvZ2Z9IiB8IHNlZCAncy9eLyAgICAvJwogICAgICAgIGlmIFsgIiR7dGllcn0iID0gInJlcSIgXTsgdGhlbgogICAgICAgICAgICBGQUlMRURfUkVRVUlSRUQrPSgiJHtuYW1lfSIpCiAgICAgICAgZWxzZQogICAgICAgICAgICBGQUlMRURfT1BUSU9OQUwrPSgiJHtuYW1lfSIpCiAgICAgICAgZmkKICAgIGZpCn0KCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIFJFUVVJUkVEIFBMVUdJTlMgKHBpcGVsaW5lIGNhbm5vdCBydW4gd2l0aG91dCB0aGVzZSkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpidWlsZF9mbXRjb252KCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9FbGVvbm9yZU1pem8vZm10Y29udi5naXQgZm10Y29udgogICAgY2QgZm10Y29udi9idWlsZC91bml4CiAgICAuL2F1dG9nZW4uc2gKICAgIC4vY29uZmlndXJlIC0tcHJlZml4PSIke1ZTUFJFRklYfSIgLS1saWJkaXI9IiR7VlNQUkVGSVh9L2xpYiIKICAgIG1ha2UgLWoiJChucHJvYykiCiAgICBpbnN0X3NvIC4KfQpidWlsZCBmbXRjb252IHJlcSBidWlsZF9mbXRjb252CgpidWlsZF9mZm1zMigpIHsKICAgIGdjbG9uZSBodHRwczovL2dpdGh1Yi5jb20vRkZNUy9mZm1zMi5naXQgZmZtczIKICAgIGNkIGZmbXMyCiAgICAuL2F1dG9nZW4uc2gKICAgIC4vY29uZmlndXJlIC0tcHJlZml4PSIke1ZTUFJFRklYfSIgLS1saWJkaXI9IiR7VlNQUkVGSVh9L2xpYiIgLS1lbmFibGUtc2hhcmVkCiAgICBtYWtlIC1qIiQobnByb2MpIgogICAgbWFrZSBpbnN0YWxsCiAgICAjIGZmbXMyIGluc3RhbGxzIGl0cyBWUyBwbHVnaW4gLnNvIGludG8gdGhlIHByZWZpeDsgYWxzbyBzdGFnZSB0byBwbHVnaW5kaXIuCiAgICBmaW5kICIke1ZTUFJFRklYfSIgLW5hbWUgJ2xpYmZmbXMyKi5zbyonIC1leGVjIGNwIC12IHt9ICIke1BMVUdJTkRJUn0vIiBcOyAyPi9kZXYvbnVsbCB8fCB0cnVlCiAgICBpbnN0X3NvIC4KfQpidWlsZCBmZm1zMiByZXEgYnVpbGRfZmZtczIKCmJ1aWxkX2xzbWFzaCgpIHsKICAgIGdjbG9uZSBodHRwczovL2dpdGh1Yi5jb20vbC1zbWFzaC9sLXNtYXNoLmdpdCBsLXNtYXNoCiAgICBjZCBsLXNtYXNoCiAgICAuL2NvbmZpZ3VyZSAtLXByZWZpeD0iJHtWU1BSRUZJWH0iIC0tZW5hYmxlLXNoYXJlZAogICAgbWFrZSAtaiIkKG5wcm9jKSIgbGliCiAgICBtYWtlIGluc3RhbGwtbGliCn0KYnVpbGQgbHNtYXNoIHJlcSBidWlsZF9sc21hc2gKCmJ1aWxkX2xzbWFzKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9Ba2FyaW5WUy9MLVNNQVNILVdvcmtzLmdpdCBsc21hcy1ha2FyaW4KICAgIGNkIGxzbWFzLWFrYXJpbi9WYXBvdXJTeW50aAogICAgbWVzb24gc2V0dXAgYnVpbGQgLS1idWlsZHR5cGU9cmVsZWFzZSAtLXByZWZpeD0iJHtWU1BSRUZJWH0iIC0tbGliZGlyPSIke1ZTUFJFRklYfS9saWIiCiAgICBuaW5qYSAtQyBidWlsZCAtaiIkKG5wcm9jKSIKICAgIGluc3Rfc28gYnVpbGQKfQpidWlsZCBsc21hcyByZXEgYnVpbGRfbHNtYXMKCmJ1aWxkX212dG9vbHMoKSB7CiAgICBnY2xvbmUgaHR0cHM6Ly9naXRodWIuY29tL2R1YmhhdGVyL3ZhcG91cnN5bnRoLW12dG9vbHMuZ2l0IG12dG9vbHMKICAgIG1lc29uX2J1aWxkIG12dG9vbHMKfQpidWlsZCBtdnRvb2xzIHJlcSBidWlsZF9tdnRvb2xzCgpidWlsZF96bmVkaTMoKSB7CiAgICAjIHpuZWRpMyB1c2VzIGEgcGxhaW4gTWFrZWZpbGUgKG5vIG1lc29uKS4gQnVpbGQgZnJvbSByb290OyB0aGUgLnNvIGxhbmRzCiAgICAjIGluIHRoZSByZXBvIHJvb3QgYXMgdnN6bmVkaTMuc28uCiAgICBnY2xvbmUgaHR0cHM6Ly9naXRodWIuY29tL3Nla3JpdC10d2Mvem5lZGkzLmdpdCB6bmVkaTMKICAgIGNkIHpuZWRpMwogICAgZ2l0IHN1Ym1vZHVsZSB1cGRhdGUgLS1pbml0IC0tcmVjdXJzaXZlCiAgICBtYWtlIC1qIiQobnByb2MpIgogICAgIyBUaGUgVlMgcGx1Z2luIC5zbyBpcyBidWlsdCBpbnRvIHRoZSByZXBvIHJvb3QKICAgIGluc3Rfc28gLgp9CmJ1aWxkIHpuZWRpMyByZXEgYnVpbGRfem5lZGkzCgpidWlsZF9ubmVkaTNjbCgpIHsKICAgIGdjbG9uZSBodHRwczovL2dpdGh1Yi5jb20vSG9tZU9mVmFwb3VyU3ludGhFdm9sdXRpb24vVmFwb3VyU3ludGgtTk5FREkzQ0wuZ2l0IG5uZWRpM2NsCiAgICBtZXNvbl9idWlsZCBubmVkaTNjbAp9CmJ1aWxkIG5uZWRpM2NsIHJlcSBidWlsZF9ubmVkaTNjbAoKYnVpbGRfa25sbSgpIHsKICAgIGdjbG9uZSBodHRwczovL2dpdGh1Yi5jb20vS2hhbmF0dGlsYS9LTkxNZWFuc0NMLmdpdCBrbmxtCiAgICBtZXNvbl9idWlsZCBrbmxtCn0KYnVpbGQga25sbSByZXEgYnVpbGRfa25sbQoKYnVpbGRfY2FzKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9Ib21lT2ZWYXBvdXJTeW50aEV2b2x1dGlvbi9WYXBvdXJTeW50aC1DQVMuZ2l0IGNhcwogICAgbWVzb25fYnVpbGQgY2FzCn0KYnVpbGQgY2FzIHJlcSBidWlsZF9jYXMKCmJ1aWxkX2N0bWYoKSB7CiAgICBnY2xvbmUgaHR0cHM6Ly9naXRodWIuY29tL0hvbWVPZlZhcG91clN5bnRoRXZvbHV0aW9uL1ZhcG91clN5bnRoLUNUTUYuZ2l0IGN0bWYKICAgIG1lc29uX2J1aWxkIGN0bWYKfQpidWlsZCBjdG1mIHJlcSBidWlsZF9jdG1mCgpidWlsZF9yZ3ZzKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS92YXBvdXJzeW50aC92cy1yZW1vdmVncmFpbi5naXQgcmd2cwogICAgbWVzb25fYnVpbGQgcmd2cwp9CmJ1aWxkIHJndnMgcmVxIGJ1aWxkX3JndnMKCmJ1aWxkX21pc2NmaWx0ZXJzKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS92YXBvdXJzeW50aC92cy1taXNjZmlsdGVycy1vYnNvbGV0ZS5naXQgbWlzYwogICAgbWVzb25fYnVpbGQgbWlzYwp9CmJ1aWxkIG1pc2NmaWx0ZXJzIHJlcSBidWlsZF9taXNjZmlsdGVycwoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgT1BUSU9OQUwgUExVR0lOUyAoZmFpbHVyZXMgbG9nZ2VkLCBidWlsZCBjb250aW51ZXMpCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKYnVpbGRfZWVkaTMoKSB7CiAgICBnY2xvbmUgaHR0cHM6Ly9naXRodWIuY29tL0hvbWVPZlZhcG91clN5bnRoRXZvbHV0aW9uL1ZhcG91clN5bnRoLUVFREkzLmdpdCBlZWRpMwogICAgbWVzb25fYnVpbGQgZWVkaTMgLURvcGVuY2w9dHJ1ZSB8fCBtZXNvbl9idWlsZCBlZWRpMwp9CmJ1aWxkIGVlZGkzbSBvcHQgYnVpbGRfZWVkaTMKCmJ1aWxkX25uZWRpMygpIHsKICAgIGdjbG9uZSBodHRwczovL2dpdGh1Yi5jb20vZHViaGF0ZXIvdmFwb3Vyc3ludGgtbm5lZGkzLmdpdCB2c25uZWRpMwogICAgbWVzb25fYnVpbGQgdnNubmVkaTMKfQpidWlsZCBubmVkaTMgb3B0IGJ1aWxkX25uZWRpMwoKYnVpbGRfYndkaWYoKSB7CiAgICBnY2xvbmUgaHR0cHM6Ly9naXRodWIuY29tL0hvbWVPZlZhcG91clN5bnRoRXZvbHV0aW9uL1ZhcG91clN5bnRoLUJ3ZGlmLmdpdCBid2RpZgogICAgbWVzb25fYnVpbGQgYndkaWYKfQpidWlsZCBid2RpZiBvcHQgYnVpbGRfYndkaWYKCmJ1aWxkX2FkZGdyYWluKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9Ib21lT2ZWYXBvdXJTeW50aEV2b2x1dGlvbi9WYXBvdXJTeW50aC1BZGRHcmFpbi5naXQgYWRkZ3JhaW4KICAgIG1lc29uX2J1aWxkIGFkZGdyYWluCn0KYnVpbGQgYWRkZ3JhaW4gb3B0IGJ1aWxkX2FkZGdyYWluCgpidWlsZF9kY3RmaWx0ZXIoKSB7CiAgICBnY2xvbmUgaHR0cHM6Ly9naXRodWIuY29tL0hvbWVPZlZhcG91clN5bnRoRXZvbHV0aW9uL1ZhcG91clN5bnRoLURDVEZpbHRlci5naXQgZGN0ZmlsdGVyCiAgICBtZXNvbl9idWlsZCBkY3RmaWx0ZXIKfQpidWlsZCBkY3RmaWx0ZXIgb3B0IGJ1aWxkX2RjdGZpbHRlcgoKYnVpbGRfZGZ0dGVzdCgpIHsKICAgIGdjbG9uZSBodHRwczovL2dpdGh1Yi5jb20vSG9tZU9mVmFwb3VyU3ludGhFdm9sdXRpb24vVmFwb3VyU3ludGgtREZUVGVzdC5naXQgZGZ0dGVzdAogICAgbWVzb25fYnVpbGQgZGZ0dGVzdAp9CmJ1aWxkIGRmdHRlc3Qgb3B0IGJ1aWxkX2RmdHRlc3QKCmJ1aWxkX3R0ZW1wc21vb3RoKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9Ib21lT2ZWYXBvdXJTeW50aEV2b2x1dGlvbi9WYXBvdXJTeW50aC1UVGVtcFNtb290aC5naXQgdHRlbXBzbW9vdGgKICAgIG1lc29uX2J1aWxkIHR0ZW1wc21vb3RoCn0KYnVpbGQgdHRlbXBzbW9vdGggb3B0IGJ1aWxkX3R0ZW1wc21vb3RoCgpidWlsZF9mbHV4c21vb3RoKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9kdWJoYXRlci92YXBvdXJzeW50aC1mbHV4c21vb3RoLmdpdCBmbHV4c21vb3RoCiAgICBtZXNvbl9idWlsZCBmbHV4c21vb3RoCn0KYnVpbGQgZmx1eHNtb290aCBvcHQgYnVpbGRfZmx1eHNtb290aAoKYnVpbGRfaHFkbjNkKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9IaW50ZXJ3YWVsZGxlcnMvdmFwb3Vyc3ludGgtaHFkbjNkLmdpdCBocWRuM2QKICAgIG1lc29uX2J1aWxkIGhxZG4zZAp9CmJ1aWxkIGhxZG4zZCBvcHQgYnVpbGRfaHFkbjNkCgpidWlsZF9kZWJsb2NrKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9Ib21lT2ZWYXBvdXJTeW50aEV2b2x1dGlvbi9WYXBvdXJTeW50aC1EZWJsb2NrLmdpdCBkZWJsb2NrCiAgICBtZXNvbl9idWlsZCBkZWJsb2NrCn0KYnVpbGQgZGVibG9jayBvcHQgYnVpbGRfZGVibG9jawoKYnVpbGRfc2FuZ25vbSgpIHsKICAgIGdjbG9uZSBodHRwczovL2dpdGh1Yi5jb20vZHViaGF0ZXIvdmFwb3Vyc3ludGgtc2FuZ25vbS5naXQgc2FuZ25vbQogICAgbWVzb25fYnVpbGQgc2FuZ25vbQp9CmJ1aWxkIHNhbmdub20gb3B0IGJ1aWxkX3Nhbmdub20KCmJ1aWxkX2F3YXJwc2hhcnAyKCkgewogICAgZ2Nsb25lIGh0dHBzOi8vZ2l0aHViLmNvbS9kdWJoYXRlci92YXBvdXJzeW50aC1hd2FycHNoYXJwMi5naXQgYXdhcnBzaGFycDIKICAgIG1lc29uX2J1aWxkIGF3YXJwc2hhcnAyCn0KYnVpbGQgYXdhcnBzaGFycDIgb3B0IGJ1aWxkX2F3YXJwc2hhcnAyCgpidWlsZF9lZWRpMigpIHsKICAgIGdjbG9uZSBodHRwczovL2dpdGh1Yi5jb20vSG9tZU9mVmFwb3VyU3ludGhFdm9sdXRpb24vVmFwb3VyU3ludGgtRUVESTIuZ2l0IGVlZGkyCiAgICBtZXNvbl9idWlsZCBlZWRpMgp9CmJ1aWxkIGVlZGkyIG9wdCBidWlsZF9lZWRpMgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgU3VtbWFyeQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KZWNobwplY2hvICIjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMiCmVjaG8gIiMjIEJVSUxEIFNVTU1BUlkiCmVjaG8gIiMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyIKZWNobyAiQnVpbHQgcGx1Z2lucyAoJHsjQlVJTFRbQF19KTogJHtCVUlMVFsqXX0iCmVjaG8KZWNobyAiT3B0aW9uYWwgZmFpbHVyZXMgKCR7I0ZBSUxFRF9PUFRJT05BTFtAXX0pOiAke0ZBSUxFRF9PUFRJT05BTFsqXTotbm9uZX0iCmVjaG8gIlJlcXVpcmVkIGZhaWx1cmVzICgkeyNGQUlMRURfUkVRVUlSRURbQF19KTogJHtGQUlMRURfUkVRVUlSRURbKl06LW5vbmV9IgplY2hvCmVjaG8gIkluc3RhbGxlZCAuc28gZmlsZXMgaW4gJHtQTFVHSU5ESVJ9OiIKbHMgLTEgIiR7UExVR0lORElSfSIgfCBzZWQgJ3MvXi8gIC8nCmVjaG8gIlRvdGFsOiAkKGxzIC0xICIke1BMVUdJTkRJUn0iIHwgd2MgLWwpIC5zbyBmaWxlcyIKCmlmIFsgIiR7I0ZBSUxFRF9SRVFVSVJFRFtAXX0iIC1ndCAwIF07IHRoZW4KICAgIGVjaG8KICAgIGVjaG8gIkZBVEFMOiAkeyNGQUlMRURfUkVRVUlSRURbQF19IFJFUVVJUkVEIHBsdWdpbihzKSBmYWlsZWQ6ICR7RkFJTEVEX1JFUVVJUkVEWypdfSIKICAgIGVjaG8gIlBlci1wbHVnaW4gbG9ncyBpbiAke0xPR0RJUn0vIgogICAgZXhpdCAxCmZpCmVjaG8gIk9LOiBhbGwgcmVxdWlyZWQgcGx1Z2lucyBidWlsdC4iCmV4aXQgMAo="
    Path("/tmp/build_plugins.sh").write_bytes(base64.b64decode(BUILD_PLUGINS_B64))
    sh("chmod +x /tmp/build_plugins.sh && bash /tmp/build_plugins.sh")

    # nnedi3_rpow2 is NOT on PyPI — ship as bundled .py module.
    sh(f"cp /content/nnedi3_rpow2.py {VSPREFIX}/py/lib/python3.12/site-packages/nnedi3_rpow2.py 2>/dev/null || true")
    sh(f"{VSPREFIX}/py/bin/pip install vsdehalo vsjetpack vsutil vstools")
    sh(f"{VSPREFIX}/py/bin/pip install 'git+https://github.com/HomeOfVapourSynthEvolution/havsfunc.git'")

    # Static FFmpeg with libx264.
    sh("cd /tmp && wget -q https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz && tar xJf ffmpeg-release-amd64-static.tar.xz")
    sh(f"cp /tmp/ffmpeg-*-static/ffmpeg {VSPREFIX}/bin/ffmpeg && cp /tmp/ffmpeg-*-static/ffprobe {VSPREFIX}/bin/ffprobe && chmod +x {VSPREFIX}/bin/ffmpeg {VSPREFIX}/bin/ffprobe")

# ---------- Set environment for every subsequent cell ----------------------
os.environ["PATH"] = f"{VSPREFIX}/py/bin:{VSPREFIX}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{VSPREFIX}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["VAPOURSYNTH_EXTRA_PLUGIN_PATH"] = f"{VSPREFIX}/plugins"
os.environ["PYTHONPATH"] = f"{VSPREFIX}/py/lib/python3.12/site-packages:" + os.environ.get("PYTHONPATH", "")

with open("/etc/profile.d/vs_env.sh", "w") as f:
    f.write(textwrap.dedent(f"""\\
        export PATH={VSPREFIX}/py/bin:{VSPREFIX}/bin:$PATH
        export LD_LIBRARY_PATH={VSPREFIX}/lib:${{LD_LIBRARY_PATH:-}}
        export VAPOURSYNTH_EXTRA_PLUGIN_PATH={VSPREFIX}/plugins
        export PYTHONPATH={VSPREFIX}/py/lib/python3.12/site-packages:${{PYTHONPATH:-}}
    """))

print("\\n=== vspipe --version ===")
sh(f"{VSPREFIX}/vspipe --version || {VSPREFIX}/py/bin/vspipe --version || {VSPREFIX}/bin/vspipe --version", check=False)
print("\\n=== ffmpeg version ===")
sh(f"{VSPREFIX}/bin/ffmpeg -version | head -1")

# vspipe needs a TOML config so VSScript can find Python.
import subprocess as _sp
VSSCRIPT_SO = _sp.run(["realpath", f"{VSPREFIX}/libvsscript.so"], capture_output=True, text=True).stdout.strip()
if not VSSCRIPT_SO:
    VSSCRIPT_SO = f"{VSPREFIX}/libvsscript.so"
# pbs's sysconfig LIBDIR reports a stale build path (/install/lib); the real
# relocatable libpython sits under the bundle. Locate it directly.
import glob as _glob
_libs = _glob.glob(f"{VSPREFIX}/py/lib/libpython3.12.so.1.0")
PYTHON_LIB = _libs[0] if _libs else f"{VSPREFIX}/py/lib/libpython3.12.so.1.0"
import os
os.makedirs(os.path.expanduser("~/.config/vapoursynth"), exist_ok=True)
with open(os.path.expanduser("~/.config/vapoursynth/vapoursynth.toml"), "w") as _f:
    _f.write('"' + VSSCRIPT_SO + '" = ["' + f"{VSPREFIX}/py/bin/python" + '", "' + PYTHON_LIB + '"]\n')
print("--- vsscript config ---")
with open(os.path.expanduser("~/.config/vapoursynth/vapoursynth.toml")) as _f:
    print(_f.read())


In [ ]:
#@title 3 - Verify ALL required plugins load (fail fast)
#@markdown This is the guard the 2021 notebook lacked. If any required plugin
#@markdown namespace is missing, we stop here -- no silent fall-back to a
#@markdown wrong-quality `yadif+hqdn3d` ffmpeg chain.
import os, subprocess, sys

VSPREFIX = os.environ.get("VSPREFIX", "/opt/vs")
PY = f"{VSPREFIX}/py/bin/python"

probe = r"""
import sys, importlib
import vapoursynth as vs
core = vs.core
loaded = sorted(p.namespace for p in core.plugins())
print(f"VapourSynth: {vs.core.core_version!s}")
print(f"Plugin count: {len(loaded)}")
for n in loaded: print("  -", n)

required = {"std","resize","lsmas","mv","znedi3","nnedi3cl",
            "fmtc","cas","knlm","ctmf","rgvs","misc"}
nice_to_have = {"nnedi3","eedi3m","bwdif","grain","dctfilter","dfttest"}
missing_required  = sorted(required  - set(loaded))
missing_nice      = sorted(nice_to_have - set(loaded))
print()
print("REQUIRED missing:", missing_required or "(none)")
print("OPTIONAL missing:", missing_nice or "(none)")

for mod in ("havsfunc", "vsdehalo", "nnedi3_rpow2"):
    try:
        importlib.import_module(mod)
        print(f"  py: {mod} OK")
    except Exception as e:
        print(f"  py: {mod} FAIL -> {e}")
        sys.exit(2)

if missing_required:
    sys.exit(1)
print("ALL REQUIRED PRESENT")
"""
r = subprocess.run([PY, "-c", probe], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print("STDERR:", r.stderr)
if r.returncode != 0:
    raise SystemExit(f"Plugin verification FAILED with code {r.returncode}. STOP.")
print("\n>>> Plugin stack OK. Proceed.")


In [ ]:
#@title 4 - Mount Drive, stage source files
#@markdown Drop your `.avi` files in:
#@markdown ```
#@markdown /content/drive/MyDrive/avis/Source_in/
#@markdown ```
#@markdown Outputs land in:
#@markdown ```
#@markdown /content/drive/MyDrive/avis/Target_out/
#@markdown ```
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

SRC_DIR  = Path('/content/drive/MyDrive/avis/Source_in')
OUT_DIR  = Path('/content/drive/MyDrive/avis/Target_out')
SRC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

sources = sorted(list(SRC_DIR.glob('*.avi')) + list(SRC_DIR.glob('*.AVI')) + list(SRC_DIR.glob('*.mov')) + list(SRC_DIR.glob('*.MOV')))
print(f"Source dir: {SRC_DIR}")
print(f"Output dir: {OUT_DIR}")
print(f"Found {len(sources)} file(s):")
for p in sources:
    sz = p.stat().st_size / (1024*1024)
    print(f"  {sz:8.1f} MB  {p.name}")

if not sources:
    print("\n*** No media files (.avi/.mov). Upload some to Source_in/ and re-run this cell. ***")


In [ ]:
#@title 5 - Write the GPU-mode aware .vpy (ported from masterpaldvcTOP.vpy)
#@markdown Modes selected at runtime by `VS_GPU_MODE` from Cell 1:
#@markdown - `opencl`: full Windows pipeline (QTGMC opencl=True, KNLMeansCL, nnedi3cl upscale).
#@markdown - `cuda`/`cpu`: QTGMC opencl=False, hqdn3d denoise, znedi3 upscale.
import os
from pathlib import Path

VPY = Path('/content/master.vpy')
VPY_BODY = r"""
# masterpaldvcTOP.vpy -- Colab port (GPU-mode aware).
# CLIP path is injected via: vspipe -a CLIP_PATH=/path/to/file.avi master.vpy ...
import os, sys
import vapoursynth as vs
core = vs.core

GPU_MODE = os.environ.get("VS_GPU_MODE", "opencl").lower()
OPENCL   = (GPU_MODE == "opencl")

import havsfunc as haf
import nnedi3_rpow2 as rpow2
import vsdehalo

core.num_threads = 8

# CLIP_PATH comes from `vspipe -a CLIP_PATH=...` (lands in module globals).
CLIP = globals().get("CLIP_PATH") or os.environ.get("CLIP_PATH")
if not CLIP:
    raise RuntimeError("CLIP_PATH not set. Pass with: vspipe -a CLIP_PATH=/path/file.avi master.vpy ...")

# ---------- SOURCE ----------------------------------------------------------
src = core.lsmas.LWLibavSource(CLIP)

# ---------- COLORSPACE: AVI rawvideo decodes RGB; go YUV444P8 (NOT 420) -----
# (See Windows .vpy comment: 444 keeps fields clean through deinterlace; chroma
# is sub-sampled to 420 LATER, on progressive frames.)
src = core.resize.Bicubic(src, format=vs.YUV444P8, matrix_s="170m",
                          dither_type="error_diffusion")

# ---------- Field order: TFF (matches Windows fix) --------------------------
src = core.std.SetFieldBased(src, 2)  # 2 = TFF

# ---------- EEDI3CL compat shim (only fires if eedi3m has no EEDI3CL) -------
def _install_eedi3cl_shim():
    try:
        core.eedi3m.EEDI3CL
        return  # already present
    except AttributeError:
        pass
    import vapoursynth as _vs
    _orig = _vs.Plugin.__getattr__
    def _shim(self, name):
        if name == "EEDI3CL":
            def _impl(clip, *a, device=None, **k):
                return core.eedi3m.EEDI3(clip, *a, **k)
            return _impl
        return _orig(self, name)
    _vs.Plugin.__getattr__ = _shim
_install_eedi3cl_shim()

# ---------- DEINTERLACE (QTGMC) ---------------------------------------------
clip = haf.QTGMC(src, Preset="Fast", TFF=True, opencl=OPENCL,
                 device=0 if OPENCL else -1)

# ---------- CROP after deinterlace (progressive, mod-2 legal) ---------------
clip = core.std.Crop(clip, right=12, bottom=6)   # 720x576 -> 708x570

# ---------- DENOISE: GPU KNLMeansCL on opencl, hqdn3d on cuda/cpu -----------
if OPENCL:
    clip = core.knlm.KNLMeansCL(clip, d=1, a=2, s=4, h=1.0,
                                channels="YUV", device_type="gpu", device_id=0)
else:
    try:
        clip = core.hqdn3d.Hqdn3d(clip, lum_spac=4, chrom_spac=3, lum_tmp=6, chrom_tmp=4.5)
    except Exception:
        pass

# ---------- HALO removal ----------------------------------------------------
try:
    clip = vsdehalo.fine_dehalo(clip, rx=2, ry=2, darkstr=0.5, brightstr=0.75)
except Exception as e:
    print(f"fine_dehalo skipped: {e}", file=sys.stderr)

# ---------- COLOR: Rec.601 -> Rec.709 (real matrix recompute via RGBS) ------
clip = core.resize.Bicubic(clip, format=vs.RGBS, matrix_in_s="170m")
clip = core.resize.Bicubic(clip, format=vs.YUV420P8, matrix_s="709")

# ---------- UPSCALE: nnedi3cl on opencl, znedi3 on cuda/cpu -----------------
upsizer = "nnedi3cl" if OPENCL else "znedi3"
clip = rpow2.nnedi3_rpow2(clip, rfactor=2, nns=4, qual=2, upsizer=upsizer,
                          width=1440, height=1080)

# ---------- CAS sharpen -----------------------------------------------------
clip = core.cas.CAS(clip, sharpness=0.4)

clip.set_output()
"""
VPY.write_text(VPY_BODY)
print(f"Wrote {VPY}")
print("GPU mode env var:", os.environ.get("VS_GPU_MODE"))


In [ ]:
#@title 6 - Encode (vspipe | ffmpeg) with audio mux
#@markdown Re-implements the Windows `convertvpy.bat` logic:
#@markdown vspipe streams y4m video on stdout; ffmpeg takes vspipe (stdin) AND
#@markdown the ORIGINAL .avi as input #2 for the audio track. Both inputs get
#@markdown `-thread_queue_size 4096` to silence the slow-pipe vs fast-disk
#@markdown blocking warning.
#@markdown
#@markdown Set `DURATION` to limit to first N seconds (smoke test); 0 = full.
ENCODER  = "libx264"   #@param ["libx264", "h264_nvenc"]
CRF_QP   = 17          #@param {type:"integer"}
DAR      = "4:3"   #@param ["4:3","16:9","81:50"]
DURATION = 5           #@param {type:"integer"}

import os, subprocess, time
from pathlib import Path

VSPREFIX = os.environ.get("VSPREFIX", "/opt/vs")
VSPIPE  = f"{VSPREFIX}/vspipe"
if not Path(VSPIPE).exists():
    VSPIPE = f"{VSPREFIX}/py/bin/vspipe"
if not Path(VSPIPE).exists():
    VSPIPE = f"{VSPREFIX}/bin/vspipe"
FFMPEG  = f"{VSPREFIX}/bin/ffmpeg"

SRC_DIR = Path('/content/drive/MyDrive/avis/Source_in')
OUT_DIR = Path('/content/drive/MyDrive/avis/Target_out')
VPY     = Path('/content/master.vpy')

sources = sorted(list(SRC_DIR.glob('*.avi')) + list(SRC_DIR.glob('*.AVI')) + list(SRC_DIR.glob('*.mov')) + list(SRC_DIR.glob('*.MOV')))
if not sources:
    raise SystemExit(f"No media sources (.avi/.mov) in {SRC_DIR}")

for i, src in enumerate(sources, 1):
    out = OUT_DIR / f"{src.stem}_restored.mp4"
    print(f"\n=== [{i}/{len(sources)}] {src.name} -> {out.name}")
    t0 = time.time()

    vp_cmd = [VSPIPE, "-c", "y4m", "-a", f"CLIP_PATH={src}", str(VPY), "-"]
    if ENCODER == "libx264":
        v_args = ["-c:v", "libx264", "-crf", str(CRF_QP), "-preset", "medium",
                  "-profile:v", "high", "-pix_fmt", "yuv420p"]
    else:
        v_args = ["-c:v", "h264_nvenc", "-qp", str(CRF_QP), "-preset", "p7",
                  "-tune", "hq", "-rc", "constqp", "-rc-lookahead", "32",
                  "-profile:v", "high", "-pix_fmt", "yuv420p"]

    ff_cmd = [
        FFMPEG, "-y",
        "-thread_queue_size", "4096", "-i", "-",
        "-thread_queue_size", "4096", "-i", str(src),
        "-map", "0:v:0", "-map", "1:a:0?",
        *(["-t", str(DURATION)] if DURATION and DURATION > 0 else []),
        *v_args,
        "-aspect", DAR,
        "-c:a", "aac", "-b:a", "192k", "-async", "1",
        str(out)
    ]

    print("VSPIPE:", " ".join(vp_cmd))
    print("FFMPEG:", " ".join(ff_cmd))

    env = os.environ.copy()
    env["CLIP_PATH"] = str(src)

    vp = subprocess.Popen(vp_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env)
    ff = subprocess.Popen(ff_cmd, stdin=vp.stdout, stderr=subprocess.STDOUT,
                          stdout=subprocess.PIPE, env=env,
                          bufsize=1, universal_newlines=True)
    vp.stdout.close()  # let ffmpeg drive backpressure on vspipe

    for line in ff.stdout:
        if line.startswith("frame=") or "time=" in line:
            print("\r" + line.rstrip()[:160], end="", flush=True)
        else:
            print(line.rstrip())
    ff.wait(); vp.wait()
    print()
    dt = time.time() - t0
    print(f"vspipe rc={vp.returncode}  ffmpeg rc={ff.returncode}  elapsed={dt/60:.1f} min")
    if vp.returncode != 0:
        print("--- vspipe stderr (tail) ---")
        print(vp.stderr.read().decode("utf-8", "replace")[-2000:])
    if ff.returncode != 0:
        raise SystemExit(f"FAILED on {src.name}")
    if out.exists():
        size_mb = out.stat().st_size / (1024*1024)
        print(f"OUTPUT: {out}  ({size_mb:.1f} MB)")


In [ ]:
#@title 7 - Verify output with ffprobe (resolution, codec, audio, A/V delta)
import os, subprocess, json
from pathlib import Path

VSPREFIX = os.environ.get("VSPREFIX", "/opt/vs")
FFPROBE  = f"{VSPREFIX}/bin/ffprobe"
OUT_DIR  = Path('/content/drive/MyDrive/avis/Target_out')

results = sorted(OUT_DIR.glob("*_restored.mp4"))
if not results:
    raise SystemExit(f"No outputs in {OUT_DIR}")

bad = 0
for out in results:
    print(f"\n=== {out.name}")
    r = subprocess.run(
        [FFPROBE, "-v", "error", "-print_format", "json",
         "-show_streams", "-show_format", str(out)],
        capture_output=True, text=True)
    if r.returncode != 0:
        print("FFPROBE FAILED:", r.stderr); bad += 1; continue
    meta = json.loads(r.stdout)
    streams = meta.get("streams", [])
    v = next((s for s in streams if s["codec_type"] == "video"), None)
    a = next((s for s in streams if s["codec_type"] == "audio"), None)
    print(f"  format  : {meta['format'].get('format_name')}  "
          f"size={int(meta['format'].get('size',0))/1024/1024:.1f} MB  "
          f"dur={float(meta['format'].get('duration',0)):.2f}s")
    if v:
        print(f"  video   : {v['codec_name']} {v['width']}x{v['height']} "
              f"pix={v.get('pix_fmt')} dar={v.get('display_aspect_ratio','?')}")
        ok_h264 = v['codec_name'] == 'h264'
        ok_dim  = (v['width'], v['height']) == (1440, 1080)
        if not (ok_h264 and ok_dim):
            print(f"  WARN: expected h264 1440x1080 (got {v['codec_name']} {v['width']}x{v['height']})")
            bad += 1
    else:
        print("  WARN: no video stream"); bad += 1
    if a:
        print(f"  audio   : {a['codec_name']} {a.get('sample_rate')}Hz {a.get('channels')}ch")
        if a['codec_name'] != 'aac':
            print(f"  WARN: expected AAC (got {a['codec_name']})"); bad += 1
    else:
        print("  WARN: no audio stream (was source video-only?)")

    if v and a and v.get("duration") and a.get("duration"):
        dv = float(v["duration"]); da = float(a["duration"])
        delta = abs(dv - da)
        print(f"  A/V dur : video={dv:.3f}s audio={da:.3f}s delta={delta:.3f}s")
        if delta > 0.1:
            print(f"  WARN: A/V delta > 0.1s"); bad += 1

if bad == 0:
    print("\n>>> All outputs PASS verification.")
else:
    print(f"\n>>> {bad} warning(s) raised. Inspect manually before declaring done.")
